In [ ]:
%pip install requests pandas pyproj geotessera umap-learn contextily --break-system-packages

In [5]:
import pandas as pd
from pyproj import Transformer

df = pd.read_csv('../data/raw/ukbmssitelocationdata2023.csv', encoding='cp1252')

bng_to_wgs84 = Transformer.from_crs("EPSG:27700", "EPSG:4326", always_xy=True)

def convert_row(row):
    if row['Country'] in ('England', 'Scotland', 'Wales', 'Northern Ireland'):
        lon, lat = bng_to_wgs84.transform(row['Easting'], row['Northing'])
        return pd.Series({'lat': lat, 'lon': lon})
    else:
        # Channel Islands, Isle of Man — local coordinates, skipped.
        return pd.Series({'lat': None, 'lon': None})

df[['lat', 'lon']] = df.apply(convert_row, axis=1)

converted = df.dropna(subset=['lat', 'lon'])
skipped = len(df) - len(converted)
print(f"Converted: {len(converted)} sites")
print(f"Skipped: {skipped}")

Converted: 5995 sites
Skipped: 57


In [6]:
# Sanity check — should be 0
out_of_range = converted[
    ~converted['lat'].between(49, 61) | ~converted['lon'].between(-9, 2)
]
print(f"Out-of-range after conversion: {len(out_of_range)} (should be 0)")

converted.to_csv('../data/processed/ukbms_sites_latlon.csv', index=False)
print("Saved -> data/processed/ukbms_sites_latlon.csv")

Out-of-range after conversion: 0 (should be 0)
Saved -> data/processed/ukbms_sites_latlon.csv


In [7]:
converted.info()

<class 'pandas.DataFrame'>
Index: 5995 entries, 0 to 6051
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Site_Number          5995 non-null   int64  
 1   Site_Name            3677 non-null   str    
 2   Gridreference        5995 non-null   str    
 3   Easting              5995 non-null   int64  
 4   Northing             5995 non-null   int64  
 5   Length               4831 non-null   float64
 6   Country              5995 non-null   str    
 7   N_sections           5508 non-null   float64
 8   N_yrs_surveyed       5995 non-null   int64  
 9   First_year_surveyed  5995 non-null   int64  
 10  Last_year_surveyed   5995 non-null   int64  
 11  Survey_type          5995 non-null   str    
 12  lat                  5995 non-null   float64
 13  lon                  5995 non-null   float64
dtypes: float64(4), int64(6), str(4)
memory usage: 882.9 KB


In [9]:
(converted.isna()).sum()

Site_Number               0
Site_Name              2318
Gridreference             0
Easting                   0
Northing                  0
Length                 1164
Country                   0
N_sections              487
N_yrs_surveyed            0
First_year_surveyed       0
Last_year_surveyed        0
Survey_type               0
lat                       0
lon                       0
dtype: int64

In [11]:
converted['Survey_type'].value_counts()

Survey_type
UKBMS    3676
WCBS     2319
Name: count, dtype: int64

In [13]:
wcbs_with_name = converted[
    (converted['Survey_type'] == 'WCBS') & converted['Site_Name'].notna()
]
print(f"\nWCBS rows WITH Site_Name: {len(wcbs_with_name)}")
print(wcbs_with_name[['Site_Number', 'Site_Name', 'Survey_type', 'Country', 'Gridreference']])


WCBS rows WITH Site_Name: 1
      Site_Number Site_Name Survey_type Country Gridreference
6022        52347    SM8112        WCBS   Wales        SM8112


In [15]:
print(converted[['lat','lon','First_year_surveyed','Last_year_surveyed']].isna().sum())

lat                    0
lon                    0
First_year_surveyed    0
Last_year_surveyed     0
dtype: int64


In [17]:
print('duplicate Site_Number:', converted.duplicated(subset='Site_Number').sum())
print('duplicate exact coords:', converted.duplicated(subset=['lat','lon']).sum())

duplicate Site_Number: 0
duplicate exact coords: 101


In [19]:
dupes = converted[converted.duplicated(subset=['lat','lon'], keep=False)].sort_values(['lat','lon'])
print(dupes[['Site_Number','Site_Name','Survey_type','Country','First_year_surveyed','Last_year_surveyed']].head(20))

      Site_Number                          Site_Name Survey_type  Country  \
844          1823                      Cape Cornwall       UKBMS  England   
900          1881          Cape Cornwall - new route       UKBMS  England   
829          1804                    Gwithian Towans       UKBMS  England   
830          1805                   Gwithian Sandpit       UKBMS  England   
768          1737            Primley wood and meadow       UKBMS  England   
778          1747  Primley Wood and Meadow extension       UKBMS  England   
3727         9001        West Down (HBFrit Transect)       UKBMS  England   
3728         9002          West Down (Frit Transect)       UKBMS  England   
754          1722                          Waterleat       UKBMS  England   
806          1780                     Waterleat 2006       UKBMS  England   
761          1729                 Bovey Heath cmpt 2       UKBMS  England   
765          1734             Bovey Heath comp 2 new       UKBMS  England   

In [20]:
# Count sites active in each year (First <= year <= Last)
year_counts = (
    converted.assign(
        years=lambda d: d.apply(
            lambda r: range(int(r["First_year_surveyed"]), int(r["Last_year_surveyed"]) + 1),
            axis=1,
        )
    )
    .explode("years")
    .groupby("years")
    .size()
    .rename("n_sites")
    .sort_index()
)

best_year = year_counts.idxmax()
best_count = year_counts.max()

print(f"Best year: {best_year} with {best_count} sites")
print(year_counts.sort_values(ascending=False).head(10))

Best year: 2022 with 2972 sites
years
2022    2972
2021    2929
2019    2903
2023    2892
2020    2806
2018    2805
2017    2706
2016    2590
2015    2504
2014    2398
Name: n_sites, dtype: int64


In [21]:
YEAR = 2022
active = converted[(converted['First_year_surveyed'] <= YEAR) & (converted['Last_year_surveyed'] >= YEAR)]
print('duplicate coords among sites active in', YEAR, ':', active.duplicated(subset=['lat','lon']).sum())

duplicate coords among sites active in 2022 : 9


In [22]:
active_deduped = active.drop_duplicates(subset=['lat','lon'], keep='first')
print(f"{len(active_deduped)} of {len(active)} sites remain after removing exact-coordinate duplicates")

2963 of 2972 sites remain after removing exact-coordinate duplicates


In [23]:
print(active_deduped.isna().sum())
print()
print(active_deduped['Survey_type'].value_counts())

Site_Number              0
Site_Name              915
Gridreference            0
Easting                  0
Northing                 0
Length                 176
Country                  0
N_sections              79
N_yrs_surveyed           0
First_year_surveyed      0
Last_year_surveyed       0
Survey_type              0
lat                      0
lon                      0
dtype: int64

Survey_type
UKBMS    2048
WCBS      915
Name: count, dtype: int64


In [24]:
active_deduped.to_csv("../data/processed/ukbms_sites_2022_ready.csv", index=False)